# Gulfstream walkthrough — yield curves (`zero_rates`)

End-to-end Graph **1** and Graph **2** on real curve data, using the public
`gulfstream` API (`run_single_segmentation`, `refine_regimes`, `plot_regimes`).

| Part | Focus |
|------|--------|
| A | PCA baseline → Graph 1 + Graph 2 |
| B | Kernel PCA → Graph 1 + Graph 2 |
| C | DMD → Graph 1 + Graph 2 |
| D | Search methods: **Binseg** / **BottomUp** (vs PELT) |
| E | Tests: **energy_distance** / **mmd_unbiased** |
| F | Data-driven window: **ESS** heuristic |
| G | Classical hard-label detectors (k-means / HMM) + Graph 2 |
| H | Classical models as soft dimred into kernel_ruptures |
| I | TFT attention embeddings as dimred (+ optional Graph 2) |
| — | Comparison (covering + breakpoint F1 vs PCA) |

**Database:** `D:/data/duckdb/ycs_data.duckdb` · **table:** `zero_rates`

Run top-to-bottom. After you finish, tell the agent so outputs can be checked.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

YCS_DB = Path(r"D:/data/duckdb/ycs_data.duckdb")
OUT_DIR = ROOT / "outputs" / "notebooks" / "ycs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("YCS_DB exists =", YCS_DB.exists())


## 1. Peek at `zero_rates`

Long format: one row per `(date, source)` with tenor columns `Y002p0`, `Y005p0`, …


In [ ]:
con = duckdb.connect(str(YCS_DB), read_only=True)
print("tables:", con.execute("SHOW TABLES").fetchall())
sample = con.execute(
    """
    SELECT date, source, Y002p0, Y005p0, Y010p0, Y030p0
    FROM zero_rates
    WHERE source IN ('USA', 'DEU', 'ITA')
      AND date >= '2015-01-01'
    ORDER BY date, source
    LIMIT 6
    """
).pl()
print(sample)
coverage = con.execute(
    """
    SELECT source, COUNT(*) AS n, MIN(date) AS dmin, MAX(date) AS dmax
    FROM zero_rates
    GROUP BY 1
    ORDER BY 1
    """
).pl()
print(coverage)
con.close()


## 2. Load through the public API

`gulfstream.load_features` reads `config/sources/notebook_ycs.yaml` (USA/DEU/ITA
tenors + FX, then yield-feature engineering).


In [ ]:
from gulfstream import load_features
from gulfstream.common import frames

features_df = load_features(
    ROOT / "config" / "sources" / "notebook_ycs.yaml",
    project_root=ROOT,
)
print("shape:", features_df.shape, "n_features:", frames.n_features(features_df))
print("date range:", features_df["date"].min(), "→", features_df["date"].max())
print("feature sample:", frames.feature_columns(features_df)[:10])
features_df.head(3)


## 3. Explore a few series


In [ ]:
plot_cols = [
    c
    for c in [
        "USA_Y010p0",
        "DEU_Y010p0",
        "ITA_Y010p0",
        "USA_Y002p0_minus_USA_Y010p0",
        "EURUSD",
    ]
    if c in features_df.columns
]
long = (
    features_df.select(["date", *plot_cols])
    .unpivot(index="date", on=plot_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long["date"] = pd.to_datetime(long["date"])

(
    ggplot(long, aes("date", "value", color="series"))
    + geom_line(size=0.4)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.2 * len(plot_cols)), legend_position="none")
    + labs(title="Selected yield / FX features", x="", y="")
)


## 4. Shared helpers (public API)

Graph 1 uses `run_single_segmentation`. Graph 2 uses `refine_regimes(..., seed=...)`,
which builds `retrain.regimes_df` from the Graph 1 `SegmentResults`.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream import (
    plot_regimes,
    refine_regimes,
    regime_intervals,
    run_single_segmentation,
    seed_regimes_from_results,
)
from gulfstream.common import frames, utils
from gulfstream.common.options import (
    ClassicalDetector,
    DetectionBackend,
    SearchMethod,
    StatTest,
)
from gulfstream.metrics.evaluation import (
    breakpoint_precision_recall_f1,
    covering_metric,
)


def load_core_params(img_dir: Path) -> dict:
    """Validated Graph 1 core params with notebook-friendly metrics."""
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred: {method}")
    return out


def with_search(params: dict, method) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["search_method"] = [str(method)]
    return out


def with_test(params: dict, choice) -> dict:
    out = copy.deepcopy(params)
    out["test"]["choice"] = [str(choice)]
    return out


def with_ess_window(
    params: dict,
    *,
    ess_fraction: float = 0.25,
    min_window: int = 20,
    max_window: int = 100,
) -> dict:
    out = copy.deepcopy(params)
    out["test"]["window"] = [
        {
            "method": "ess",
            "ess_fraction": ess_fraction,
            "min_window": min_window,
            "max_window": max_window,
        }
    ]
    return out


def with_classical(
    params: dict,
    detector,
    *,
    regimes: int | None = 3,
    min_regime_length: int = 20,
    **algo_extras,
) -> dict:
    """Hard-label classical backend (former --mode legacy)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.CLASSICAL)]
    out["algo"]["regime_detection_algorithm"] = [str(detector)]
    out["algo"]["dimred"] = ["raw"]
    out["algo"]["feature_map_approx_method"] = ["raw"]
    out["algo"]["post_processing_method"] = ["majority_voting"]
    out["algo"]["min_regime_length"] = [min_regime_length]
    out["algo"]["include_last_regime"] = [True]
    if regimes is not None:
        out["algo"]["regimes"] = [regimes]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_model_dimred(params: dict, method, *, regimes: int = 3) -> dict:
    """Use classical models as soft embeddings into kernel_ruptures."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = [str(method)]
    out["algo"]["regimes"] = [regimes]
    return out


def with_tft(
    params: dict,
    *,
    rank: int = 8,
    encoder_length: int = 20,
    prediction_length: int = 5,
    max_epochs: int = 1,
    batch_size: int = 16,
    mode: str = "multivariate",
) -> dict:
    """TFT attention embeddings → kernel_ruptures (smoke-friendly defaults)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = ["tft"]
    out["algo"]["rank"] = [rank]
    out["algo"]["rank_selection_method"] = ["user_specified"]
    out["algo"]["tft_encoder_length"] = [encoder_length]
    out["algo"]["tft_prediction_length"] = [prediction_length]
    out["algo"]["tft_max_epochs"] = [max_epochs]
    out["algo"]["tft_batch_size"] = [batch_size]
    out["algo"]["tft_mode"] = [mode]
    # Keep the ruptures grid small — TFT itself is the expensive step.
    out["algo"]["num_features"] = [30]
    out["algo"]["depth"] = [1]
    return out


def summarize(res, label: str, df: pl.DataFrame) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def run_g1(df: pl.DataFrame, params: dict, label: str, plot_vars: list[str]):
    """Graph 1 via public single-pass API (fast, returns SegmentResults)."""
    print(f"=== Graph 1 · {label} · backend={params['algo'].get('detection_backend')} "
          f"dimred={params['algo']['dimred']} "
          f"detector={params['algo'].get('regime_detection_algorithm')} "
          f"search={params['algo'].get('search_method')} "
          f"test={params['test'].get('choice')} ===")
    proc = run_single_segmentation(df, params)
    summarize(proc, label, df)
    plot_regimes(df, proc, variables=plot_vars[:2], title=f"Graph 1 · {label}", mode="display")
    return proc


def run_g2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    plot_vars: list[str],
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
) -> Path:
    """Graph 2 via refine_regimes, seeded from a Graph 1 SegmentResults."""
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "regimes_df": None,  # refine_regimes fills from seed=
    }
    print(f"=== Graph 2 · {label} · seeding from Graph 1 ===")
    print(seed_regimes_from_results(df, seed_res).to_dicts())
    refined = refine_regimes(df, g2, seed=seed_res)
    if refined is not None:
        summarize(refined, f"{label} Graph 2", df)
        plot_regimes(
            df,
            refined,
            variables=plot_vars[:2],
            title=f"Graph 2 · {label}",
            mode="display",
        )
    pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))[:6]
    print(f"Graph 2 artifacts under {out_dir}")
    for p in pngs:
        print(" ", p.relative_to(out_dir))
        try:
            display(Image(filename=str(p)))
        except Exception as exc:
            print("  (could not display)", exc)
    return out_dir


print("Helpers ready: run_g1/g2, with_dimred/search/test/ess/classical/model_dimred/tft")
print("Enums:", list(DetectionBackend), list(ClassicalDetector)[:4], "...")


---
# Part A — PCA (baseline)

Default core: **PCA → RFF → PELT → MMD**, then Graph 2 seeded from that run.


## A.1 Graph 1 (PCA + PELT)


In [ ]:
params_pca = with_dimred(load_core_params(OUT_DIR / "pca"), "pca")
params_pca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_pca = run_g1(features_df, params_pca, "PCA", plot_cols)
regime_intervals(proc_pca, frames.dates_series(features_df).to_list())


## A.2 Graph 2 (seeded from PCA)


In [ ]:
g2_pca_dir = run_g2(
    features_df, params_pca, proc_pca, OUT_DIR / "pca" / "graph2", "PCA", plot_cols, max_iter=3
)


---
# Part B — Kernel PCA


## B.1 Graph 1 (kPCA)


In [ ]:
params_kpca = with_dimred(load_core_params(OUT_DIR / "kpca"), "kpca")
params_kpca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_kpca = run_g1(features_df, params_kpca, "kPCA", plot_cols)


## B.2 Graph 2 (seeded from kPCA)


In [ ]:
g2_kpca_dir = run_g2(
    features_df, params_kpca, proc_kpca, OUT_DIR / "kpca" / "graph2", "kPCA", plot_cols, max_iter=3
)


---
# Part C — DMD


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = with_dimred(load_core_params(OUT_DIR / "dmd"), "dmd")
params_dmd["metrics"]["features_to_plot"] = plot_cols[:3]
proc_dmd = run_g1(features_df, params_dmd, "DMD", plot_cols)


## C.2 Graph 2 (seeded from DMD)


In [ ]:
g2_dmd_dir = run_g2(
    features_df, params_dmd, proc_dmd, OUT_DIR / "dmd" / "graph2", "DMD", plot_cols, max_iter=3
)


---
# Part D — Search methods (Binseg / BottomUp)

Same PCA embedding and MMD test as Part A; only `algo.search_method` changes.
Compare candidate-generation strategies against the PELT baseline.


## D.1 Binseg


In [ ]:
params_binseg = with_search(
    with_dimred(load_core_params(OUT_DIR / "binseg"), "pca"),
    SearchMethod.BINSEG,
)
params_binseg["metrics"]["features_to_plot"] = plot_cols[:3]
proc_binseg = run_g1(features_df, params_binseg, "Binseg", plot_cols)


## D.2 BottomUp


In [ ]:
params_bottomup = with_search(
    with_dimred(load_core_params(OUT_DIR / "bottomup"), "pca"),
    SearchMethod.BOTTOMUP,
)
params_bottomup["metrics"]["features_to_plot"] = plot_cols[:3]
proc_bottomup = run_g1(features_df, params_bottomup, "BottomUp", plot_cols)


---
# Part E — Statistical tests

Same PCA + PELT search as Part A; swap `test.choice` to **energy distance**
(no kernel bandwidth) and **unbiased MMD**.


## E.1 Energy distance


In [ ]:
params_energy = with_test(
    with_dimred(load_core_params(OUT_DIR / "energy"), "pca"),
    StatTest.ENERGY_DISTANCE,
)
params_energy["metrics"]["features_to_plot"] = plot_cols[:3]
proc_energy = run_g1(features_df, params_energy, "energy_distance", plot_cols)


## E.2 Unbiased MMD


In [ ]:
params_mmd_u = with_test(
    with_dimred(load_core_params(OUT_DIR / "mmd_unbiased"), "pca"),
    StatTest.MMD_UNBIASED,
)
params_mmd_u["metrics"]["features_to_plot"] = plot_cols[:3]
proc_mmd_u = run_g1(features_df, params_mmd_u, "mmd_unbiased", plot_cols)


---
# Part F — ESS window

Replace the fixed MMD window with the **effective-sample-size** heuristic
(`test.window.method: ess`). Search and test stay at PCA + PELT + MMD.


In [ ]:
params_ess = with_ess_window(
    with_dimred(load_core_params(OUT_DIR / "ess"), "pca"),
    ess_fraction=0.25,
    min_window=20,
    max_window=100,
)
params_ess["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ess = run_g1(features_df, params_ess, "ESS window", plot_cols)


---
# Part G — Classical hard-label detectors

Former **legacy** mode is now Graph 1 with `algo.detection_backend: classical`.
Detectors assign labels → breakpoints (no RFF / PELT / MMD). Same public API:
`run_single_segmentation` / `refine_regimes`.


## G.1 k-means (classical)


In [ ]:
params_ckmeans = with_classical(
    load_core_params(OUT_DIR / "classical_kmeans"),
    ClassicalDetector.KMEANS,
    regimes=3,
    random_state=42,
)
params_ckmeans["metrics"]["features_to_plot"] = plot_cols[:3]
proc_ckmeans = run_g1(features_df, params_ckmeans, "classical kmeans", plot_cols)


## G.2 HMM (classical)


In [ ]:
params_chmm = with_classical(
    load_core_params(OUT_DIR / "classical_hmm"),
    ClassicalDetector.HMM,
    regimes=3,
    hmm_emissions="gaussian",
    hmm_n_iter=50,
)
params_chmm["metrics"]["features_to_plot"] = plot_cols[:3]
proc_chmm = run_g1(features_df, params_chmm, "classical HMM", plot_cols)


## G.3 Graph 2 seeded from classical k-means


In [ ]:
g2_ckmeans_dir = run_g2(
    features_df,
    params_ckmeans,
    proc_ckmeans,
    OUT_DIR / "classical_kmeans" / "graph2",
    "classical kmeans",
    plot_cols,
    max_iter=2,
)


---
# Part H — Classical models as soft dimred

Same algorithm families as Part G, but as **embeddings** into the default
`kernel_ruptures` stack (`algo.dimred: [kmeans|hmm]` → RFF → PELT → MMD).
This is how Graph 1 used “legacy” models without leaving the ruptures path.


## H.1 k-means dimred → kernel ruptures


In [ ]:
params_kmeans_dim = with_model_dimred(
    load_core_params(OUT_DIR / "kmeans_dimred"),
    "kmeans",
    regimes=3,
)
params_kmeans_dim["metrics"]["features_to_plot"] = plot_cols[:3]
proc_kmeans_dim = run_g1(features_df, params_kmeans_dim, "kmeans dimred", plot_cols)


## H.2 HMM dimred → kernel ruptures


In [ ]:
params_hmm_dim = with_model_dimred(
    load_core_params(OUT_DIR / "hmm_dimred"),
    "hmm",
    regimes=3,
)
params_hmm_dim["metrics"]["features_to_plot"] = plot_cols[:3]
proc_hmm_dim = run_g1(features_df, params_hmm_dim, "HMM dimred", plot_cols)


---
# Part I — TFT dimred (Temporal Fusion Transformer)

Train a short TFT and use its **attention vectors** as the Graph 1 embedding
(`algo.dimred: [tft]` → RFF → PELT → MMD). Needs `torch`, `lightning`, and
`pytorch-forecasting`.

Notebook defaults are smoke settings (1 epoch, encoder=20). Prefer
**multivariate** mode on wide panels; univariate melts every feature into its
own series and is much slower.


## I.1 Graph 1 (TFT attention embeddings)


In [ ]:
# Skip cleanly if the optional TFT stack is missing.
try:
    import torch  # noqa: F401
    import lightning  # noqa: F401
    import pytorch_forecasting  # noqa: F401
    _TFT_OK = True
except ImportError as exc:
    _TFT_OK = False
    print("TFT stack unavailable — skipping Part I:", exc)

if _TFT_OK:
    params_tft = with_tft(load_core_params(OUT_DIR / "tft"))
    params_tft["metrics"]["features_to_plot"] = plot_cols[:3]
    print(
        "TFT smoke:",
        f"n={features_df.height}",
        f"enc={params_tft['algo']['tft_encoder_length']}",
        f"epochs={params_tft['algo']['tft_max_epochs']}",
        f"mode={params_tft['algo']['tft_mode']}",
    )
    proc_tft = run_g1(features_df, params_tft, "TFT dimred", plot_cols)
else:
    proc_tft = proc_pca  # placeholder so the comparison cell still runs


## I.2 Graph 2 seeded from TFT (optional, still expensive)


In [ ]:
if _TFT_OK:
    g2_tft_dir = run_g2(
        features_df,
        params_tft,
        proc_tft,
        OUT_DIR / "tft" / "graph2",
        "TFT",
        plot_cols,
        max_iter=1,
    )
else:
    print("Skipping TFT Graph 2")


---
# Comparison

Covering and breakpoint F1 (tolerance = 10 days) against the **PCA / PELT / MMD**
baseline from Part A.


In [ ]:
dates = frames.dates_series(features_df).to_list()
baseline = proc_pca
n = features_df.height

def row(label: str, res) -> dict:
    f1 = breakpoint_precision_recall_f1(baseline.bkpts, res.bkpts, tolerance=10)
    return {
        "run": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "dates": [str(dates[b]) for b in res.bkpts],
        "covering_vs_pca": covering_metric(baseline.bkpts, res.bkpts, n),
        "f1_vs_pca": f1["f1"],
        "precision_vs_pca": f1["precision"],
        "recall_vs_pca": f1["recall"],
    }

summary = pl.DataFrame(
    [
        row("A pca/pelt/mmd", proc_pca),
        row("B kpca", proc_kpca),
        row("C dmd", proc_dmd),
        row("D binseg", proc_binseg),
        row("D bottomup", proc_bottomup),
        row("E energy", proc_energy),
        row("E mmd_unbiased", proc_mmd_u),
        row("F ess window", proc_ess),
        row("G classical kmeans", proc_ckmeans),
        row("G classical hmm", proc_chmm),
        row("H kmeans dimred", proc_kmeans_dim),
        row("H hmm dimred", proc_hmm_dim),
        row("I tft dimred", proc_tft),
    ]
)
summary


## CLI equivalents

```bash
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/default_core.yaml \
  --source-config config/sources/notebook_ycs.yaml

uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/full_graph2.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Classical hard-label (former legacy mode)
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/classical_kmeans.yaml \
  --source-config config/sources/notebook_ycs.yaml

# Soft dimred with classical models / TFT
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_kmeans_dimred.yaml \
  --source-config config/sources/notebook_ycs.yaml
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_tft_dimred.yaml \
  --source-config config/sources/notebook_ycs.yaml
```

Set `algo.search_method`, `test.choice`, `test.window`,
`algo.detection_backend: [classical]`, or `algo.dimred: [tft]` to match Parts D–I.
